In [ ]:
# config.py
import torch
import os
import matplotlib.pyplot as plt
from PIL import Image

# --- Training Hyperparameters ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE_GEN = 1e-4
LEARNING_RATE_DISC = 1e-4
BATCH_SIZE = 16
NUM_EPOCHS_PRETRAIN = 50
NUM_EPOCHS_GAN = 100
HIGH_RES_SIZE = 96  # As per the paper
LOW_RES_SIZE = HIGH_RES_SIZE // 4
NUM_WORKERS = 4
LAMBDA_VGG = 1.0  # Weight for VGG/content loss
LAMBDA_ADV = 1e-3  # Weight for adversarial loss

os.makedirs("saved_models", exist_ok=True)

# --- Model Paths ---
PRETRAINED_GEN_PATH = "saved_models/srresnet_pretrained.pth"
GEN_PATH = "saved_models/generator.pth"
DISC_PATH = "saved_models/discriminator.pth"

# --- Dataset Paths ---
TRAIN_DIR = "/root/.cache/kagglehub/datasets/takihasan/div2k-dataset-for-super-resolution/versions/1/Dataset/DIV2K_train_HR"
TEST_DIR = "/root/.cache/kagglehub/datasets/takihasan/div2k-dataset-for-super-resolution/versions/1/Dataset/DIV2K_valid_HR"

# set the repo name
model_name = "keanteng/srgan-div2k-0723-v2"

In [ ]:
from torch import nn

class Generator(nn.Module):
    """
    The Generator Network (SRResNet).
    It takes a low-resolution image and outputs a super-resolved version.
    """
    def __init__(self, in_channels=3, num_res_blocks=16):
        super(Generator, self).__init__()
        self.initial_conv = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=9, stride=1, padding=4),
            nn.PReLU()
        )

        self.residuals = nn.Sequential(*[ResidualBlock(64) for _ in range(num_res_blocks)])

        self.mid_conv = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64)
        )

        # Upsampling by 4x (two 2x upsample blocks)
        self.upsample_blocks = nn.Sequential(
            UpsampleBlock(64),
            UpsampleBlock(64),
        )

        self.final_conv = nn.Conv2d(64, in_channels, kernel_size=9, stride=1, padding=4)

    def forward(self, x):
        initial_out = self.initial_conv(x)
        residual_out = self.residuals(initial_out)
        mid_out = self.mid_conv(residual_out)
        mid_out = mid_out + initial_out # Skip connection
        upsampled_out = self.upsample_blocks(mid_out)
        final_out = self.final_conv(upsampled_out)
        return torch.tanh(final_out) # Tanh activation to scale output to [-1, 1], can be scaled back for display also
    # note for input 

In [ ]:
# dataset.py
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class ImageDataset(Dataset):
    """
    Custom dataset to load high-resolution images and create low-resolution counterparts.
    """
    def __init__(self, hr_dir, hr_size):
        super(ImageDataset, self).__init__()
        self.hr_image_files = [os.path.join(hr_dir, f) for f in os.listdir(hr_dir)]
        self.hr_size = hr_size

        # Transform for the original image before cropping
        self.initial_transform = transforms.Compose([
            transforms.ToTensor(), # lr image go through here will be scaled to [0, 1]
        ])

        # Normalization transforms
        self.hr_normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # normalize to [-1, 1]
        # take 0 and 1 range
        # 0: (0-0.5)/0.5 = -1
        # 1: (1-0.5)/0.5 = 1
        self.lr_normalize = transforms.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0]) # nothing change x = (x - mean) / std, so if mean=0 and std=1, x remains unchanged, we will use tanh to scale to [-1,1] in the generator

    # called by pytorch DataLoader
    # get a single item for each dataset using the index
    def __getitem__(self, index):
        # Load image
        hr_image = Image.open(self.hr_image_files[index]).convert("RGB")

        # Convert to tensor first
        hr_tensor = self.initial_transform(hr_image)

        # Apply random crop to get consistent size
        crop_transform = transforms.RandomCrop(self.hr_size)
        hr_cropped = crop_transform(hr_tensor)

        # Create LR version by downsampling the cropped HR image
        lr_tensor = transforms.functional.resize(
            hr_cropped,
            size=self.hr_size // 4,
            interpolation=transforms.InterpolationMode.BICUBIC # when u upscale, fill the neighboring pixel values
            # when downscale, it will use the average of the neighboring pixel values
        )

        # Apply normalization
        hr_normalized = self.hr_normalize(hr_cropped)
        lr_normalized = self.lr_normalize(lr_tensor)

        return lr_normalized, hr_normalized

    # called by pytorch DataLoader
    # return the length of the dataset
    # SO HOW MANY IMAGES ARE THERE IN THE DATASET?
    def __len__(self):
        return len(self.hr_image_files)

# From The Paper

In [ ]:
# train_srresnet.py
import torch
from torch import optim, nn
from torch.utils.data import DataLoader
from tqdm import tqdm
#import config
#from models import Generator
#from dataset import ImageDataset

def train_srresnet():
    # load the data by giving the directory and size
    dataset = ImageDataset(hr_dir=TRAIN_DIR, hr_size=HIGH_RES_SIZE)
    # for dataloader, it will get the lenght of dataset from the class ImageDataset __len__
    # then it will iterate through the dataset to get the batches
    loader = DataLoader(
        dataset, # from the dataset class
        batch_size=BATCH_SIZE, # if 2 then 2 images will be loaded at a time, if 15 then 15 images will be loaded at a time
        # shuffle at once or batch by batch?
        # shuffle everything first say 1,2,3,4,5,6,7,8,9,10
        # become 4,1,3,2,5,6,8,7,9,10 then batch by batch
        # batch 1: 4,1
        # batch 2: 3,2 and so on
        shuffle=True, # shuffle dataset order, why? prevent learning from the order of training example
        num_workers=NUM_WORKERS,
        # number of cpu cores availabe where multiple sample can be loaded
        # let's say 4 cores, then 4 samples can be loaded at a time
        # worker 1 first sample, worker 2 second sample, worker 3 third sample, worker 4 fourth sample
        # and so on
        pin_memory=True
        # pin memory is used to speed up the transfer of data to the GPU
        # it will copy the data to the pinned memory before transferring to the GPU
    )

    # call the Generator model
    gen = Generator().to(DEVICE)
    # create an optimizer for the generator
    opt_gen = optim.Adam(gen.parameters(), lr=LEARNING_RATE_GEN)
    # what is this Adam doing here?
    # adjust the generator weights to reduce the loss
    # my words:
    # So let's say one loop is done, generator will backpropogate to update the weight, and the weight should updated using optimized loss
    mse_loss = nn.MSELoss()
    # this is euclidean distance
    # we train SRRESNET-MSE so the loss is euclidean distance between 
    # the generated high-resolution image and the actual high-resolution image

    gen.train()
    # this is training mode, we also have .eval() for evaluation mode
    # in this mode, dropout and batch normalization will behave differently
    # so for activations it will randomly zero out some of it
    # so what the heck is going on here?
    # my word: for dropout you make certain tensor zero
    # so what layers are affected by this?
    # how many tensors are affected by this?
    # say we have 128 features with dropout 0.5, then 64 features will be zeroed out randomly
    # my word: dropout after activation function like ReLU or PReLU, and so on
    # for batch normalization it will use the current mini batch statistics
    # my word: For batch norm what do you mean by current mini batch, it is say we have batch size 16 so we will use this 16 samples? Yes

    print("--- Starting SRResNet Pre-training ---")
    # now loop through according to the number of epochs
    # so for each epoch, we will go through the entire dataset
    # what the heck is going on here
    # let's say the batch is 16 so starting here we compute the loss and update the weights
    # then another batch of 16, when all data is done start next epochs
    for epoch in range(NUM_EPOCHS_PRETRAIN):

        loop = tqdm(loader, leave=True)
        # start a progress bar for the loop
        # leave=True means the progress bar will not be cleared after the loop is done
        # why the loader can be put here?
        # my word: because the loader is iterable, it will return batches of data
        # so as we see below we loop through all the batches in the loader
        # then we only go to the outer loop of next epoch
        # my word: so loader = Dataloader() then for item in loader:
        # the following code loader will go through batch by batch until the whole dataset passes through the model right for one epoch
        total_loss = 0
        # this is the total loss for the epoch, we will average it later
        for lr, hr in loop:
            lr = lr.to(DEVICE)
            hr = hr.to(DEVICE)
            # pin the images to the device
            # so the lr and hr images are now on the device (GPU or CPU)

            gen_hr = gen(lr)
            # generate the high-resolution image from the low-resolution image
            loss = mse_loss(gen_hr, hr)
            # find the euclidean distance between the generated high-resolution image and the actual high-resolution image

            opt_gen.zero_grad()
            # clear parameters of the generator to 0
            # why do we do that? gradient are accumulated by default in PyTorch
            # so if we don't clear it, the gradients will be added to the previous gradients from previous batch
            loss.backward()
            # find out how much each parameter need to be adjust to reduce loss
            # why reduce loss?
            # my word: we want to minimize the loss, so we get close to the ground truth
            # so here do calculation
            opt_gen.step()
            # update the parameters of the generator using the optimizer
            # use the calculated gradients to adjust the weights
            # so here we do update

            total_loss += loss.item()
            # accumulate the loss for the epoch
            loop.set_postfix(loss=loss.item())
            # update the progress bar with the current loss
            # what is this postfix?
            # my word: postfix is a dictionary that will be displayed in the progress bar
            # we for each batch we can see a loss change and change and change until the end of the epoch

        avg_loss = total_loss / len(loader)
        # average the loss for the epoch
        # so we take the total loss and divide by the number of batches
        # the lenght of dataloader is the number of batches
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS_PRETRAIN}] - Avg Loss: {avg_loss:.4f}")
        # print the average loss for the epoch

        torch.save(gen.state_dict(), PRETRAINED_GEN_PATH)
        # save the generator model state dict
        # what is this state dict?
        # my word: state dict is a dictionary that maps each layer to its parameters
        # what is inside the state dict?
        # my word: the state dict contains the weights and biases of each layer
        # is it the weigh of the layer?
        # my word: yes, the weights and biases of each layer are stored in the state
    print("--- Finished SRResNet Pre-training ---")